# HAR Model Analysis — Group 6, DSE/QF4211

**Models:** HAR-RV (3 params) | HAR-RV-J (4 params) | HAR-RV-J-H (6 params)  
**Horizons:** h = 1, 3, 5, 7  
**Evaluation:** Rolling-window OOS with daily refit  

**Structure:**
- Part A — Data Loading & HAR Regressor Construction
- Part B — Benchmark Model (HAR-RV)
- Part C — Extended Models (HAR-RV-J, HAR-RV-J-H) + DM Tests
- Part D — Coefficient Evolution
- Part E — Forecast Error Decomposition
- Part F — Residual Diagnostics (Ljung-Box)
- Part G — Combined Summary & Degradation Table

## Imports

In [ ]:
from math import erf, sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
%matplotlib inline

## Part A: Data Loading & HAR Regressor Construction

Loads `full_df.csv` and constructs the three HAR regressors from the `log_RV` series:

| Regressor | Definition | Time Scale | Interpretation |
|-----------|-----------|------------|----------------|
| `har_d` | `log_RV` shifted by 1 day | Daily | Yesterday's volatility (day traders) |
| `har_w` | 5-day rolling mean of `log_RV`, shifted by 1 | Weekly | Past week average (portfolio managers) |
| `har_m` | 22-day rolling mean of `log_RV`, shifted by 1 | Monthly | Past month average (institutions) |

These follow the Heterogeneous Market Hypothesis (Corsi, 2009).

**Train/test split:** Training up to 2024-06-28, test from 2024-06-29 — matching the group's agreed evaluation framework.

In [ ]:
# ── Load data ──
CSV_PATH = Path('../data/full_df.csv')  # change this to your path

raw_df = pd.read_csv(CSV_PATH)
raw_df['date'] = pd.to_datetime(raw_df['date'])
raw_df = raw_df.sort_values('date').reset_index(drop=True)

# ── Train/test split by date (matching group convention) ──
TRAIN_END = pd.Timestamp('2024-06-28')
TEST_START = pd.Timestamp('2024-06-29')

train_df = raw_df[raw_df['date'] <= TRAIN_END].reset_index(drop=True)
test_df  = raw_df[raw_df['date'] >= TEST_START].reset_index(drop=True)

# ── Build base frame and HAR regressors ──
base_cols = ['date', 'log_RV', 'y_h1', 'y_h3', 'y_h5', 'y_h7']
full_df = pd.concat([train_df[base_cols], test_df[base_cols]], ignore_index=True)
full_df = full_df.sort_values('date').reset_index(drop=True)

full_df['har_d'] = full_df['log_RV'].shift(1)
full_df['har_w'] = full_df['log_RV'].rolling(window=5).mean().shift(1)
full_df['har_m'] = full_df['log_RV'].rolling(window=22).mean().shift(1)

feature_cols = ['har_d', 'har_w', 'har_m']
target_cols = ['y_h1', 'y_h3', 'y_h5', 'y_h7']

har_df = full_df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)

# ── Define boundaries ──
initial_start = train_df['date'].min()
initial_end   = train_df['date'].max()
oos_start     = test_df['date'].min()

train_har = har_df[(har_df['date'] >= initial_start) & (har_df['date'] <= initial_end)].copy()
test_har  = har_df[har_df['date'] >= oos_start].copy()

rolling_window = len(train_har)
oos_start_idx  = har_df.index[har_df['date'] >= oos_start][0]

print(f'Training:  {train_har["date"].min().date()} -> {train_har["date"].max().date()} ({len(train_har)} rows)')
print(f'Test:      {test_har["date"].min().date()} -> {test_har["date"].max().date()} ({len(test_har)} rows)')
print(f'Window:    {rolling_window} observations')
print()
print('HAR regressor statistics:')
har_df[feature_cols].describe().loc[['mean','std','min','max']]

## Part B: Benchmark Model — HAR-RV

Standard OLS with three regressors. The rolling window refits OLS at every forecast date.
No hyperparameters — just re-estimated coefficients.

**Metrics:** RMSE, MAE, R², QLIKE (Patton, 2011)

In [ ]:
# ── Helper functions ──

def qlike_from_log_rv(y_true_log, y_pred_log, eps=1e-12):
    """QLIKE loss — standard volatility forecast loss (Patton 2011)."""
    actual_rv = np.exp(y_true_log)
    pred_rv = np.clip(np.exp(y_pred_log), eps, None)
    return np.mean(np.log(pred_rv) + actual_rv / pred_rv)


def fit_model(train_frame, feature_cols, target_col, estimator):
    """Fit a fresh clone of the estimator."""
    model = clone(estimator)
    model.fit(train_frame[feature_cols], train_frame[target_col])
    return model


def rolling_window_forecast(full_frame, feature_cols, target_col,
                            oos_start_idx, window_size, estimator, model_name):
    """Core rolling-origin forecast loop."""
    rows = []
    for row_idx in range(oos_start_idx, len(full_frame)):
        train_slice = full_frame.iloc[
            max(0, row_idx - window_size):row_idx
        ].dropna(subset=feature_cols + [target_col])

        if len(train_slice) < window_size:
            continue

        current_row = full_frame.iloc[[row_idx]]
        model = fit_model(train_slice, feature_cols, target_col, estimator)
        pred = model.predict(current_row[feature_cols])[0]

        rows.append({
            'date': current_row['date'].iloc[0],
            'model': model_name,
            'target': target_col,
            'actual': current_row[target_col].iloc[0],
            'predicted': pred,
        })
    return pd.DataFrame(rows)


def diebold_mariano_test(loss_a, loss_b, h=1):
    """DM test with Newey-West correction. Positive stat = model A has larger losses."""
    loss_diff = np.asarray(loss_a) - np.asarray(loss_b)
    n = len(loss_diff)
    mean_diff = loss_diff.mean()
    centered = loss_diff - mean_diff
    nw_var = np.mean(centered ** 2)
    for lag in range(1, max(h, 1) + 1):
        if lag >= n: break
        autocov = np.mean(centered[lag:] * centered[:-lag])
        nw_var += 2 * (1 - lag / (max(h, 1) + 1)) * autocov
    statistic = mean_diff / np.sqrt(max(nw_var / n, 1e-12))
    p_value = 2 * (1 - 0.5 * (1 + erf(abs(statistic) / sqrt(2))))
    return statistic, p_value

In [ ]:
# ── Run HAR-RV benchmark ──
benchmark_estimator = LinearRegression()
benchmark_frames = []
benchmark_metrics = []

for target in target_cols:
    preds = rolling_window_forecast(
        har_df, feature_cols, target, oos_start_idx,
        rolling_window, benchmark_estimator, 'HAR-RV'
    )
    benchmark_frames.append(preds)
    pred_arr = preds['predicted'].to_numpy()
    actual_arr = preds['actual'].to_numpy()
    benchmark_metrics.append({
        'model': 'HAR-RV', 'target': target,
        'rmse': mean_squared_error(actual_arr, pred_arr) ** 0.5,
        'mae': mean_absolute_error(actual_arr, pred_arr),
        'r2': r2_score(actual_arr, pred_arr),
        'qlike': qlike_from_log_rv(actual_arr, pred_arr),
        'n': len(preds),
    })
    print(f'  {target}: {len(preds)} forecasts')

predictions_df = pd.concat(benchmark_frames, ignore_index=True)
benchmark_metrics_df = pd.DataFrame(benchmark_metrics)
print()
benchmark_metrics_df

In [ ]:
# ── HAR-RV forecast plots ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
fig.suptitle('HAR-RV — Forecast vs Actual', fontsize=14, fontweight='bold')
for ax, target in zip(axes.flat, target_cols):
    sub = predictions_df[predictions_df['target'] == target]
    ax.plot(sub['date'], sub['actual'], label='Actual', linewidth=1.2, color='steelblue')
    ax.plot(sub['date'], sub['predicted'], label='Predicted', linewidth=1.2,
            linestyle='--', color='coral')
    ax.set_title(target)
    ax.grid(alpha=0.3)
axes[0,0].legend()
plt.tight_layout()
plt.show()

## Part C: Extended Models — HAR-RV-J and HAR-RV-J-H

Adds jump components to test whether modelling volatility jumps improves accuracy.

**Jump proxy:** `max(log_RV - rolling_5day_mean(log_RV), 0)` — captures days where volatility exceeds its recent level.

> **Note:** The standard literature uses bipower variation (BPV) for jump detection. Our proxy is a simpler approximation.

| Model | Features | Parameters |
|-------|----------|------------|
| HAR-RV-J | har_d, har_w, har_m, **jump_d** | 4 |
| HAR-RV-J-H | har_d, har_w, har_m, **jump_d, jump_w, jump_m** | 6 |

In [ ]:
# ── Construct jump proxy ──
har_df['jump_proxy'] = np.maximum(
    har_df['log_RV'] - har_df['log_RV'].rolling(window=5).mean(), 0.0
)
har_df['jump_d'] = har_df['jump_proxy'].shift(1)
har_df['jump_w'] = har_df['jump_proxy'].rolling(window=5).mean().shift(1)
har_df['jump_m'] = har_df['jump_proxy'].rolling(window=22).mean().shift(1)

extended_cols = feature_cols + ['jump_d', 'jump_w', 'jump_m']
extended_har_df = har_df.dropna(subset=extended_cols + target_cols).reset_index(drop=True)
extended_oos_start_idx = extended_har_df.index[extended_har_df['date'] >= oos_start][0]
extended_rolling_window = len(extended_har_df[extended_har_df['date'] <= initial_end])

jp = extended_har_df['jump_proxy']
print(f'Extended dataset: {len(extended_har_df)} rows')
print(f'Rolling window:  {extended_rolling_window}')
print(f'Jump proxy: mean={jp.mean():.4f}, std={jp.std():.4f}, non-zero={(jp>0).mean()*100:.1f}%')

In [ ]:
# ── Run extended models ──
extended_model_specs = {
    'HAR-RV-J':   {'estimator': LinearRegression(), 'feature_cols': feature_cols + ['jump_d']},
    'HAR-RV-J-H': {'estimator': LinearRegression(), 'feature_cols': feature_cols + ['jump_d', 'jump_w', 'jump_m']},
}

extended_frames = []
extended_metrics = []

for model_name, spec in extended_model_specs.items():
    print(f'Running {model_name}...')
    for target in target_cols:
        preds = rolling_window_forecast(
            extended_har_df, spec['feature_cols'], target,
            extended_oos_start_idx, extended_rolling_window,
            spec['estimator'], model_name
        )
        extended_frames.append(preds)
        pred_arr = preds['predicted'].to_numpy()
        actual_arr = preds['actual'].to_numpy()
        extended_metrics.append({
            'model': model_name, 'target': target,
            'rmse': mean_squared_error(actual_arr, pred_arr) ** 0.5,
            'mae': mean_absolute_error(actual_arr, pred_arr),
            'r2': r2_score(actual_arr, pred_arr),
            'qlike': qlike_from_log_rv(actual_arr, pred_arr),
            'n': len(preds),
        })
        print(f'  {target}: {len(preds)} forecasts')

extended_predictions_df = pd.concat(extended_frames, ignore_index=True)
extended_metrics_df = pd.DataFrame(extended_metrics).sort_values(['model','target']).reset_index(drop=True)
print()
extended_metrics_df

In [ ]:
# ── Diebold-Mariano tests: all pairwise comparisons ──
all_preds = pd.concat([predictions_df, extended_predictions_df], ignore_index=True)

dm_pairs = [
    ('HAR-RV', 'HAR-RV-J'),
    ('HAR-RV', 'HAR-RV-J-H'),
    ('HAR-RV-J', 'HAR-RV-J-H'),
]

dm_results = []
for model_a, model_b in dm_pairs:
    for target in target_cols:
        a_p = all_preds[(all_preds['model'] == model_a) & (all_preds['target'] == target)]
        b_p = all_preds[(all_preds['model'] == model_b) & (all_preds['target'] == target)]
        comp = a_p.merge(b_p, on='date', suffixes=('_a', '_b'))
        if len(comp) == 0: continue
        loss_a = (comp['actual_a'] - comp['predicted_a']) ** 2
        loss_b = (comp['actual_b'] - comp['predicted_b']) ** 2
        h_val = int(target.split('h')[-1])
        dm_stat, p_val = diebold_mariano_test(loss_a, loss_b, h=h_val)
        dm_results.append({
            'comparison': f'{model_a} vs {model_b}',
            'target': target, 'DM_stat': round(dm_stat, 4),
            'p_value': round(p_val, 4),
            'sig_5pct': 'Yes' if p_val < 0.05 else 'No',
            'favours': model_b if dm_stat > 0 else model_a,
        })

dm_df = pd.DataFrame(dm_results)
dm_df

In [ ]:
# ── Extended model forecast plots ──
for model_name in extended_model_specs:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
    fig.suptitle(f'{model_name} — Forecast vs Actual', fontsize=14, fontweight='bold')
    for ax, target in zip(axes.flat, target_cols):
        sub = extended_predictions_df[
            (extended_predictions_df['model'] == model_name)
            & (extended_predictions_df['target'] == target)
        ]
        ax.plot(sub['date'], sub['actual'], label='Actual', linewidth=1.2, color='steelblue')
        ax.plot(sub['date'], sub['predicted'], label='Predicted', linewidth=1.2,
                linestyle='--', color='coral')
        ax.set_title(target)
        ax.grid(alpha=0.3)
    axes[0,0].legend()
    plt.tight_layout()
    plt.show()

## Part D: Coefficient Evolution

Tracks how much weight the HAR model puts on each of its three inputs at every forecast date.

Shows whether the model relies on different inputs during different market conditions,
and whether the weighting changes between h=1 (short horizon) and h=7 (long horizon).

- `beta_monthly` dominating → forecast driven by long-run volatility regime
- `beta_daily` spiking → short-term shocks becoming important (market stress)
- Negative `beta_weekly` at h=7 → mean-reversion mechanism

In [ ]:
# ── Track coefficients at every forecast date for h=1 and h=7 ──
coef_records = []

for target in ['y_h1', 'y_h7']:
    for row_idx in range(oos_start_idx, len(har_df)):
        train_slice = har_df.iloc[
            max(0, row_idx - rolling_window):row_idx
        ].dropna(subset=feature_cols + [target])
        if len(train_slice) < rolling_window:
            continue
        mdl = LinearRegression()
        mdl.fit(train_slice[feature_cols], train_slice[target])
        coef_records.append({
            'date': har_df.iloc[row_idx]['date'],
            'target': target,
            'intercept': mdl.intercept_,
            'beta_daily': mdl.coef_[0],
            'beta_weekly': mdl.coef_[1],
            'beta_monthly': mdl.coef_[2],
        })

coef_df = pd.DataFrame(coef_records)
print(f'Collected {len(coef_records)} coefficient snapshots')

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
fig.suptitle('HAR-RV — Coefficient Evolution', fontsize=14, fontweight='bold')
for ax, target in zip(axes, ['y_h1', 'y_h7']):
    sub = coef_df[coef_df['target'] == target]
    ax.plot(sub['date'], sub['beta_daily'], label='beta_daily', linewidth=1.2, color='#e74c3c')
    ax.plot(sub['date'], sub['beta_weekly'], label='beta_weekly', linewidth=1.2, color='#2ecc71')
    ax.plot(sub['date'], sub['beta_monthly'], label='beta_monthly', linewidth=1.2, color='#3498db')
    ax.set_title(f'Target: {target}')
    ax.set_xlabel('Date')
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel('Coefficient value')
plt.tight_layout()
plt.show()

# ── Summary stats ──
for target in ['y_h1', 'y_h7']:
    sub = coef_df[coef_df['target'] == target]
    print(f'\nTarget: {target}')
    display(sub[['intercept','beta_daily','beta_weekly','beta_monthly']].describe().loc[['mean','std','min','max']])

## Part E: Forecast Error Decomposition

Plots squared forecast errors over time for all three models and all four horizons.
Shows **when** errors happen rather than just aggregate RMSE.

Use the top 10 worst dates to cross-reference with actual BTC/macro events
(crashes, Fed decisions, liquidation events, geopolitical shocks).

In [ ]:
# ── Squared error plots for all three models ──
all_model_preds = pd.concat([predictions_df, extended_predictions_df], ignore_index=True)
model_names = ['HAR-RV', 'HAR-RV-J', 'HAR-RV-J-H']

fig, axes = plt.subplots(3, 1, figsize=(18, 15))
fig.suptitle('Squared Forecast Errors Over Time', fontsize=14, fontweight='bold')

for ax, model_name in zip(axes, model_names):
    for target in target_cols:
        sub = all_model_preds[
            (all_model_preds['model'] == model_name)
            & (all_model_preds['target'] == target)
        ].copy().sort_values('date')
        sub['sq_error'] = (sub['actual'] - sub['predicted']) ** 2
        ax.plot(sub['date'], sub['sq_error'], label=target, linewidth=0.9, alpha=0.8)
    ax.set_title(model_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Squared Error')
    ax.legend()
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 10 worst forecast dates (HAR-RV) ──
for target_label, target in [('h=1', 'y_h1'), ('h=7', 'y_h7')]:
    worst = predictions_df[predictions_df['target'] == target].copy()
    worst['sq_error'] = (worst['actual'] - worst['predicted']) ** 2
    top10 = worst.nlargest(10, 'sq_error')[['date', 'actual', 'predicted', 'sq_error']].reset_index(drop=True)
    print(f'\nTOP 10 WORST FORECAST DATES (HAR-RV, {target_label}):')
    display(top10)

## Part F: Residual Diagnostics — Ljung-Box Test

Tests whether HAR's forecast errors are **random** or contain **predictable patterns**.

- If errors have a pattern (e.g. underpredicts today → underpredicts tomorrow), the model is leaving exploitable structure behind
- If all HAR variants show significant autocorrelation, the limitation is HAR as a framework, not which variant you pick
- This directly motivates the ML models: proves there's something left for Ridge/XGBoost/LSTM to capture

**Two tests per model per horizon:**
- Raw residuals → linear autocorrelation
- Squared residuals → remaining ARCH effects (volatility clustering in the errors)

In [ ]:
# ── Ljung-Box implementation (statsmodels not available) ──

def _log_gamma(x):
    """Log-gamma function via Stirling's approximation."""
    if x <= 0: return 0.0
    if x < 7: return _log_gamma(x + 1) - np.log(x)
    return ((x - 0.5) * np.log(x) - x + 0.5 * np.log(2 * np.pi)
            + 1 / (12 * x) - 1 / (360 * x ** 3))


def chi2_survival(x, k):
    """Survival function of chi-squared distribution (1 - CDF)."""
    if x <= 0: return 1.0
    if k <= 0: return 0.0
    a = k / 2.0
    z = x / 2.0
    total = 0.0
    term = 1.0 / a
    total = term
    for n in range(1, 500):
        term *= z / (a + n)
        total += term
        if abs(term) < 1e-12: break
    log_p = a * np.log(z) - z + np.log(total) - _log_gamma(a)
    p_lower = min(max(np.exp(log_p), 0.0), 1.0)
    return 1.0 - p_lower


def manual_ljung_box(series, max_lag):
    """
    Ljung-Box test for autocorrelation.
    Large Q + small p = residuals are NOT random = model is missing something.
    """
    x = np.asarray(series)
    n = len(x)
    x_centered = x - x.mean()
    var = np.sum(x_centered ** 2) / n
    autocorrs = []
    for k in range(1, max_lag + 1):
        if k >= n:
            autocorrs.append(0.0)
            continue
        autocov = np.sum(x_centered[k:] * x_centered[:-k]) / n
        autocorrs.append(autocov / var if var > 0 else 0.0)
    Q = n * (n + 2) * sum(
        autocorrs[k] ** 2 / (n - (k + 1))
        for k in range(max_lag) if (n - (k + 1)) > 0
    )
    return Q, chi2_survival(Q, max_lag), autocorrs

In [ ]:
# ── Run Ljung-Box on all models and horizons ──
lb_results = []

for mn in model_names:
    for target in target_cols:
        sub = all_model_preds[
            (all_model_preds['model'] == mn) & (all_model_preds['target'] == target)
        ].copy().sort_values('date').reset_index(drop=True)
        if len(sub) < 30: continue

        residuals = (sub['actual'] - sub['predicted']).values
        h_val = int(target.split('h')[-1])
        test_lags = max(10, 2 * h_val)

        q_raw, p_raw, _ = manual_ljung_box(residuals, test_lags)
        q_sq, p_sq, _ = manual_ljung_box(residuals ** 2, test_lags)

        lb_results.append({
            'model': mn, 'target': target, 'lags': test_lags,
            'Q_raw': round(q_raw, 1), 'p_raw': round(p_raw, 4),
            'sig_raw': 'Yes' if p_raw < 0.05 else 'No',
            'Q_squared': round(q_sq, 1), 'p_squared': round(p_sq, 4),
            'sig_squared': 'Yes' if p_sq < 0.05 else 'No',
        })

lb_df = pd.DataFrame(lb_results)
print('Ljung-Box Test Results')
print('Yes = significant at 5% = residuals have predictable patterns')
print('= HAR is leaving exploitable structure for ML to capture')
print()
lb_df

In [ ]:
# ── ACF plots: HAR-RV residuals at h=1 vs h=7 ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('HAR-RV Residual Autocorrelation', fontsize=14, fontweight='bold')

for col_idx, target in enumerate(['y_h1', 'y_h7']):
    sub = all_model_preds[
        (all_model_preds['model'] == 'HAR-RV') & (all_model_preds['target'] == target)
    ].sort_values('date')
    resid = (sub['actual'] - sub['predicted']).values
    n = len(resid)
    _, _, acf_raw = manual_ljung_box(resid, 20)
    _, _, acf_sq = manual_ljung_box(resid ** 2, 20)
    ci = 1.96 / np.sqrt(n)

    # Raw residual ACF
    ax = axes[0, col_idx]
    ax.bar(range(1, 21), acf_raw, color='steelblue', width=0.5)
    ax.axhline(y=ci, color='red', linestyle='--', linewidth=0.8)
    ax.axhline(y=-ci, color='red', linestyle='--', linewidth=0.8)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(f'{target} — Raw Residual ACF')
    ax.set_xlabel('Lag')
    ax.grid(alpha=0.3)

    # Squared residual ACF
    ax = axes[1, col_idx]
    ax.bar(range(1, 21), acf_sq, color='coral', width=0.5)
    ax.axhline(y=ci, color='red', linestyle='--', linewidth=0.8)
    ax.axhline(y=-ci, color='red', linestyle='--', linewidth=0.8)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(f'{target} — Squared Residual ACF')
    ax.set_xlabel('Lag')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part G: Combined Summary & Degradation Table

The degradation table is the core finding: complexity helps at short horizons but hurts at long horizons.
The Ljung-Box results prove all HAR variants leave predictable structure behind — motivating the ML models.

In [ ]:
# ── Complete model comparison table ──
all_metrics = pd.concat([benchmark_metrics_df, extended_metrics_df], ignore_index=True)

for target in target_cols:
    h_val = target.split('h')[-1]
    print(f'\nHorizon h = {h_val}:')
    sub = all_metrics[all_metrics['target'] == target].sort_values('rmse')
    display(sub[['model','rmse','mae','r2','qlike']].reset_index(drop=True))

In [ ]:
# ── RMSE degradation table ──
print('RMSE DEGRADATION FROM h=1 TO h=7:')
print('=' * 60)

deg_rows = []
for mn in ['HAR-RV', 'HAR-RV-J', 'HAR-RV-J-H']:
    m = all_metrics[all_metrics['model'] == mn]
    r1 = m[m['target'] == 'y_h1']['rmse'].values[0]
    r7 = m[m['target'] == 'y_h7']['rmse'].values[0]
    delta = r7 - r1
    pct = (delta / r1) * 100
    deg_rows.append({'model': mn, 'RMSE_h1': round(r1,6), 'RMSE_h7': round(r7,6),
                     'delta': round(delta,6), 'pct_change': f'{pct:.1f}%'})

deg_df = pd.DataFrame(deg_rows)
deg_df

In [ ]:
# ── Save prediction CSVs for cross-model DM tests ──
from pathlib import Path
OUTPUT_DIR = Path('prediction_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

for target in target_cols:
    sub = predictions_df[predictions_df['target'] == target][['date','actual','predicted']].copy()
    sub.to_csv(OUTPUT_DIR / f'har_rv_{target}_predictions.csv', index=False)

for mn in ['HAR-RV-J', 'HAR-RV-J-H']:
    for target in target_cols:
        sub = extended_predictions_df[
            (extended_predictions_df['model'] == mn) & (extended_predictions_df['target'] == target)
        ][['date','actual','predicted']].copy()
        fname = f"{mn.lower().replace('-','_')}_{target}_predictions.csv"
        sub.to_csv(OUTPUT_DIR / fname, index=False)

print('Saved prediction CSVs to prediction_outputs/')
print('Done.')